In [8]:
import jax
import jax.numpy as jnp

def radix_sort_float(x):
  """
  Performs a stable radix sort on an array of JAX floats (float32 or bfloat16).

  Args:
    x: A JAX array of dtype float32 or bfloat16.

  Returns:
    A new array containing the sorted elements of x.
  """
  # Determine the integer types corresponding to the float type
  if x.dtype == jnp.float32:
    int_dtype = jnp.int32
    uint_dtype = jnp.uint32
    num_bits = 32
  elif x.dtype == jnp.bfloat16:
    # bfloat16 is actually stored as float32 with truncated mantissa
    # We need to convert to float32 first, then treat as int32
    int_dtype = jnp.int32
    uint_dtype = jnp.uint32
    num_bits = 32
  else:
    raise TypeError(f"Unsupported dtype: {x.dtype}. Must be float32 or bfloat16.")

  # --- 1. Forward Transformation (Float -> Sortable Integer) ---
  # For bfloat16, we need to convert to float32 first to get proper bit representation
  if x.dtype == jnp.bfloat16:
    x_as_float32 = x.astype(jnp.float32)
    as_int = jax.lax.bitcast_convert_type(x_as_float32, int_dtype)
  else:
    as_int = jax.lax.bitcast_convert_type(x, int_dtype)
  
  # Create the sign bit mask as a JAX unsigned integer scalar
  sign_bit_mask = jnp.array(1 << (num_bits - 1), dtype=uint_dtype)

  # Transform the integer representation to be lexicographically sortable
  transformed_int = jnp.where(as_int < 0, ~as_int, as_int ^ sign_bit_mask)

  # --- 2. Integer Radix Sort ---
  def sort_for_bit(i, current_array):
    bits = (current_array >> i) & 1
    zeros_count = current_array.size - jnp.sum(bits)
    zeros_rank = jnp.cumsum(1 - bits) - 1
    ones_rank = jnp.cumsum(bits) - 1
    new_indices = jnp.where(bits == 0, zeros_rank, zeros_count + ones_rank)
    return jnp.zeros_like(current_array).at[new_indices].set(current_array)
  
  sorted_transformed = jax.lax.fori_loop(0, num_bits, sort_for_bit, transformed_int)

  # --- 3. Inverse Transformation (Sorted Integer -> Float) ---
  inversed_int = jnp.where(
      (sorted_transformed & sign_bit_mask) != 0,
      sorted_transformed ^ sign_bit_mask,
      ~sorted_transformed
  )
  
  # Convert back to the original float type
  if x.dtype == jnp.bfloat16:
    result_as_float32 = jax.lax.bitcast_convert_type(inversed_int, jnp.float32)
    return result_as_float32.astype(jnp.bfloat16)
  else:
    return jax.lax.bitcast_convert_type(inversed_int, x.dtype)

# JIT-compile the function for performance
radix_sort_float_jit = jax.jit(radix_sort_float)

# --- Verification ---
key = jax.random.PRNGKey(0)
test_f32 = jax.random.uniform(key, (20,), minval=-100.0, maxval=100.0, dtype=jnp.float32)

print("Running corrected code...")
sorted_f32 = radix_sort_float_jit(test_f32)
is_correct = jnp.all(sorted_f32 == jnp.sort(test_f32))

print(f"Verification against jnp.sort: {is_correct}")

Running corrected code...
Verification against jnp.sort: True


In [9]:
# Create a PRNG key
key = jax.random.PRNGKey(0)

# --- Test with float32 ---
print("--- Testing float32 ---")
f32_key, bf16_key = jax.random.split(key)
test_f32 = jax.random.uniform(f32_key, (20,), minval=-100.0, maxval=100.0, dtype=jnp.float32)

print("Original f32 array:", jnp.round(test_f32, 2))
sorted_f32 = radix_sort_float_jit(test_f32)
print("Sorted f32 array:  ", jnp.round(sorted_f32, 2))
print(f"Verification: {jnp.all(sorted_f32 == jnp.sort(test_f32))}")

# --- Test with bfloat16 ---
print("\n--- Testing bfloat16 ---")
test_bf16 = jax.random.uniform(bf16_key, (20,), minval=-100.0, maxval=100.0, dtype=jnp.bfloat16)

print("Original bf16 array:", test_bf16)
sorted_bf16 = radix_sort_float_jit(test_bf16)
print("Sorted bf16 array:  ", sorted_bf16)
print(f"Verification: {jnp.all(sorted_bf16 == jnp.sort(test_bf16))}")

# --- Additional test with edge cases ---
print("\n--- Testing edge cases ---")
edge_cases_f32 = jnp.array([jnp.inf, -jnp.inf, 0.0, -0.0, jnp.nan, 1.0, -1.0], dtype=jnp.float32)
edge_cases_bf16 = jnp.array([jnp.inf, -jnp.inf, 0.0, -0.0, 1.0, -1.0], dtype=jnp.bfloat16)

print("Edge cases f32:", edge_cases_f32)
sorted_edge_f32 = radix_sort_float_jit(edge_cases_f32)
print("Sorted edge f32:", sorted_edge_f32)

print("Edge cases bf16:", edge_cases_bf16)
sorted_edge_bf16 = radix_sort_float_jit(edge_cases_bf16)
print("Sorted edge bf16:", sorted_edge_bf16)

--- Testing float32 ---
Original f32 array: [-51.37       -93.72        69.479996    19.46       -82.159996
 -88.829994     0.47        46.73        17.77       -83.89
  93.34       -70.04       -45.649998    -0.96999997  83.56
  23.16         1.25       -42.309998   -49.719997   -33.219997  ]
Sorted f32 array:   [-93.72       -88.829994   -83.89       -82.159996   -70.04
 -51.37       -49.719997   -45.649998   -42.309998   -33.219997
  -0.96999997   0.47         1.25        17.77        19.46
  23.16        46.73        69.479996    83.56        93.34      ]
Verification: True

--- Testing bfloat16 ---
Original bf16 array: [-84.5 -23.5 6 -86 -62.5 -72 -59.5 -97 -9.5 61 59 95 -97 -76.5 88 75 -61
 -12.5 -75 -67]
Sorted bf16 array:   [-97 -97 -86 -84.5 -76.5 -75 -72 -67 -62.5 -61 -59.5 -23.5 -12.5 -9.5 6 59
 61 75 88 95]
Verification: True

--- Testing edge cases ---
Edge cases f32: [ inf -inf   0.  -0.  nan   1.  -1.]
Sorted edge f32: [-inf  -1.  -0.   0.   1.  inf  nan]
Edge cases bf16

In [10]:
import time

# --- Performance comparison on large bfloat16 arrays ---
print("--- Performance comparison on 1M bfloat16 elements ---")

# Create a large random array
key = jax.random.PRNGKey(42)
large_bf16 = jax.random.uniform(key, (1000000,), minval=-1000.0, maxval=1000.0, dtype=jnp.bfloat16)

# Warm up JIT compilation
print("Warming up JIT compilation...")
_ = radix_sort_float_jit(large_bf16[:1000])
_ = jnp.sort(large_bf16[:1000])

# Time radix sort
print("Timing radix sort...")
start_time = time.time()
sorted_radix = radix_sort_float_jit(large_bf16)
radix_time = time.time() - start_time

# Time jnp.sort
print("Timing jnp.sort...")
start_time = time.time()
sorted_jnp = jnp.sort(large_bf16)
jnp_time = time.time() - start_time

# Verify correctness
is_correct = jnp.all(sorted_radix == sorted_jnp)

print(f"\nResults:")
print(f"Radix sort time: {radix_time:.4f} seconds")
print(f"jnp.sort time:   {jnp_time:.4f} seconds")
print(f"Speedup:         {jnp_time/radix_time:.2f}x")
print(f"Correctness:     {is_correct}")

# Test with different array sizes for scaling analysis
print(f"\n--- Scaling analysis ---")
sizes = [1000, 10000, 100000, 1000000]
for size in sizes:
    test_array = large_bf16[:size]
    
    # Time radix sort
    start = time.time()
    _ = radix_sort_float_jit(test_array)
    radix_t = time.time() - start
    
    # Time jnp.sort
    start = time.time()
    _ = jnp.sort(test_array)
    jnp_t = time.time() - start
    
    print(f"Size {size:>7}: Radix {radix_t:.4f}s, jnp.sort {jnp_t:.4f}s, Speedup {jnp_t/radix_t:.2f}x")

--- Performance comparison on 1M bfloat16 elements ---
Warming up JIT compilation...
Timing radix sort...
Timing jnp.sort...

Results:
Radix sort time: 0.0815 seconds
jnp.sort time:   0.0003 seconds
Speedup:         0.00x
Correctness:     True

--- Scaling analysis ---
Size    1000: Radix 0.0000s, jnp.sort 0.0000s, Speedup 0.39x
Size   10000: Radix 0.0592s, jnp.sort 0.0000s, Speedup 0.00x
Size  100000: Radix 0.0689s, jnp.sort 0.0000s, Speedup 0.00x
Size 1000000: Radix 0.0000s, jnp.sort 0.0000s, Speedup 0.69x


In [11]:
def heapsort_float(x):
    """
    In-place heapsort implementation using O(log N) additional memory.
    
    Args:
        x: A JAX array of floats to sort
        
    Returns:
        A new sorted array (JAX arrays are immutable, so we simulate in-place)
    """
    # Work with a copy since JAX arrays are immutable
    arr = x.copy()
    n = len(arr)
    
    def heapify(arr, n, i):
        """Heapify subtree rooted at index i (max heap)"""
        largest = i
        left = 2 * i + 1
        right = 2 * i + 2
        
        # If left child exists and is greater than root
        if left < n and arr[left] > arr[largest]:
            largest = left
            
        # If right child exists and is greater than current largest
        if right < n and arr[right] > arr[largest]:
            largest = right
            
        # If largest is not root, swap and continue heapifying
        if largest != i:
            arr = arr.at[i].set(arr[largest])
            arr = arr.at[largest].set(arr[i])
            arr = heapify(arr, n, largest)
            
        return arr
    
    # Build max heap (rearrange array)
    for i in range(n // 2 - 1, -1, -1):
        arr = heapify(arr, n, i)
    
    # Extract elements from heap one by one
    for i in range(n - 1, 0, -1):
        # Move current root to end
        arr = arr.at[0].set(arr[i])
        arr = arr.at[i].set(arr[0])
        
        # Call heapify on reduced heap
        arr = heapify(arr, i, 0)
    
    return arr

# JIT compile for performance
heapsort_float_jit = jax.jit(heapsort_float)

# Test the implementation
print("--- Testing O(log N) memory heapsort ---")
test_data = jax.random.uniform(jax.random.PRNGKey(123), (20,), dtype=jnp.float32)
print("Original:", jnp.round(test_data, 2))

sorted_heap = heapsort_float_jit(test_data)
sorted_reference = jnp.sort(test_data)

print("Heapsort:", jnp.round(sorted_heap, 2))
print("Reference:", jnp.round(sorted_reference, 2))
print("Correct:", jnp.allclose(sorted_heap, sorted_reference))

--- Testing O(log N) memory heapsort ---
Original: [0.58       0.08       0.65999997 0.41       0.06       0.90999997
 0.62       0.91999996 0.71999997 0.45       0.98999995 0.94
 0.55       0.31       0.22999999 0.45       0.78       0.24
 0.24       0.83      ]


TracerBoolConversionError: Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function heapsort_float at /var/folders/s2/5cjtb0gj0p96kmpbdl_l7_zc0000gn/T/ipykernel_94110/1441226008.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://jax.readthedocs.io/en/latest/errors.html#jax.errors.TracerBoolConversionError

In [ ]:
# Memory usage comparison
print("\n--- Memory usage analysis ---")

def estimate_memory_usage(sort_func, size):
    """Estimate memory usage by tracking array allocations"""
    test_arr = jax.random.uniform(jax.random.PRNGKey(0), (size,), dtype=jnp.float32)
    
    # The original radix sort creates multiple temporary arrays
    # Heapsort works in-place (modulo JAX immutability)
    
    if sort_func == radix_sort_float_jit:
        # Radix sort memory: input + transformed_int + sorted_transformed + multiple temp arrays
        # Roughly 4-5x input size
        return f"~{4 * size * 4}bytes (4x input + temporaries)"
    elif sort_func == heapsort_float_jit:
        # Heapsort memory: input + copy for in-place operations
        # Just 2x input size
        return f"~{2 * size * 4}bytes (2x input only)"
    else:
        # jnp.sort likely uses similar memory to radix sort
        return f"~{3 * size * 4}bytes (estimated)"

sizes = [1000, 10000, 100000, 1000000]
print("Size      | Radix Sort Memory    | Heapsort Memory     | jnp.sort Memory")
print("-" * 75)
for size in sizes:
    radix_mem = estimate_memory_usage(radix_sort_float_jit, size)
    heap_mem = estimate_memory_usage(heapsort_float_jit, size)
    jnp_mem = estimate_memory_usage(jnp.sort, size)
    print(f"{size:>8} | {radix_mem:<19} | {heap_mem:<18} | {jnp_mem}")

# Performance comparison on large arrays
print("\n--- Performance comparison: Memory vs Speed ---")
large_test = jax.random.uniform(jax.random.PRNGKey(999), (100000,), dtype=jnp.float32)

# Warm up
_ = heapsort_float_jit(large_test[:1000])
_ = radix_sort_float_jit(large_test[:1000])

# Time heapsort
start = time.time()
sorted_heap = heapsort_float_jit(large_test)
heap_time = time.time() - start

# Time radix sort  
start = time.time()
sorted_radix = radix_sort_float_jit(large_test)
radix_time = time.time() - start

# Time jnp.sort
start = time.time()
sorted_jnp = jnp.sort(large_test)
jnp_time = time.time() - start

print(f"Heapsort (O(log N) memory): {heap_time:.4f}s")
print(f"Radix sort (O(N) memory):   {radix_time:.4f}s") 
print(f"jnp.sort:                   {jnp_time:.4f}s")

# Verify all produce same results
print(f"All results match: {jnp.allclose(sorted_heap, sorted_jnp) and jnp.allclose(sorted_radix, sorted_jnp)}")